In [1]:
!pip -q install -U ultralytics onnx onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 81.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 77.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.0 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
from pathlib import Path
from collections import Counter

import json
import random
import shutil
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from IPython.display import Video, display
from matplotlib.patches import Rectangle
from PIL import Image
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor

In [5]:
DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike")
DATASET_ROOT  = Path("/content/RDD2022_China_MotorBike_1")


def copy_file(src_file, src_root, dst_root):
    relative_path = src_file.relative_to(src_root)
    dest_file = dst_root / relative_path
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_file, dest_file)


for split in ["train", "val"]:
    src_dir = DRIVE_DATASET_ROOT / split
    dst_dir = DATASET_ROOT / split
    files = [p for p in src_dir.rglob("*") if p.is_file()]
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = [
            executor.submit(copy_file, f, src_dir, dst_dir) for f in files
        ]
        for future in futures:
            future.result()

In [9]:
BEST_MODEL_PATH = Path(
    DRIVE_DATASET_ROOT /
    "runs/yolo26n_rdd2022_baseline" /
    "weights/best.pt"
)

DATA_YAML_PATH = Path(
    DRIVE_DATASET_ROOT / "configs" /
    "rdd2022_china_motorbike.yaml"
)

DATASET_ROOT = Path(
    "/content/"
    "RDD2022_China_MotorBike_1"
)

VAL_IMAGES_DIR = DATASET_ROOT / "val" / "images"
VAL_LABELS_DIR = DATASET_ROOT / "val" / "labels"

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/"
    "Yolo_road2022_dataset/"
    "unit_01_final_analysis"
)

required_paths = [BEST_MODEL_PATH, DATA_YAML_PATH, VAL_IMAGES_DIR, VAL_LABELS_DIR]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(f"Path not found:\n{path}")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Best model :", BEST_MODEL_PATH)
print("Validation :", VAL_IMAGES_DIR)
print("Output     :", OUTPUT_ROOT)

Best model : /content/drive/MyDrive/Yolo_road2022_dataset/RDD2022_China_MotorBike/runs/yolo26n_rdd2022_baseline/weights/best.pt
Validation : /content/RDD2022_China_MotorBike_1/val/images
Output     : /content/drive/MyDrive/Yolo_road2022_dataset/unit_01_final_analysis


**Load model and device**

In [10]:
DEVICE = (0 if torch.cuda.is_available() else "cpu")
model = YOLO(str(BEST_MODEL_PATH))

print("Device:", DEVICE)
print("Classes:")
for class_id, class_name in model.names.items():
    print(class_id, "→", class_name)

Device: 0
Classes:
0 → D00
1 → D10
2 → D20
3 → D40


In [11]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp"}

val_images = sorted(
    path
    for path in VAL_IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_SUFFIXES
)

labels_by_stem = {
    path.stem: path for path in VAL_LABELS_DIR.glob("*.txt")
}

images_by_stem = {
    path.stem: path for path in val_images
}

print("Validation images:", len(val_images))

Validation images: 387


### Systematic Error Analysis

**YOLO label → absolute XYXY**

In [12]:
def load_ground_truth(label_path, image_width, image_height):
    text = label_path.read_text(encoding="utf-8").strip()
    if not text:
        return {
            "boxes" : np.empty((0, 4), dtype=np.float32),
            "classes" : np.empty((0), dtype=np.float64)
        }
    
    boxes = []
    classes = []
    
    for line in text.splitlines():
        parts = line.split()
        class_id = int(parts[0])
        
        center_x = float(parts[1])
        center_y = float(parts[2])
        box_width = float(parts[3])
        box_height = float(parts[4])
        
        center_x *= image_width
        center_y *= image_height
        box_width *= image_width
        box_height *= image_height
        
        x1 = center_x - box_width / 2
        y1 = center_y - box_height / 2
        x2 = center_x + box_width / 2
        y2 = center_y + box_height / 2
        
        boxes.append([x1, y1, x2, y2])
        classes.append(class_id)
        
        return {
            "boxes": np.asarray(boxes, dtype=np.float32),
            "classes": np.asarray(classes,dtype=np.int64)
        }